In [45]:
import pandas as pd
import sklearn
from sklearn.pipeline import make_pipeline

df=pd.read_csv('../datasets/UCI_Credit_Card.csv')
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [46]:
df["default.payment.next.month"].value_counts(normalize=True)

,proportion
default.payment.next.month,
0,0.7788
1,0.2212


In [47]:
df.shape

(30000, 25)

In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_AMT2                   30000 non-null

In [49]:
df.isna().sum()

,0
ID,0
LIMIT_BAL,0
SEX,0
EDUCATION,0
MARRIAGE,0
AGE,0
PAY_0,0
PAY_2,0
PAY_3,0
PAY_4,0


| Famille | Colonnes | Signification |
| :--- | :--- | :--- |
| Identifiant| ID          | ID dans la BD               |
| Credit  | LIMIT_BAL         | Montant du pret accorde.               |
| Donnees demographiques et profil du client | SEX, EDUCATION, MARRIAGE, AGE | Profil du client |
| Status de remboursement | PAY_0, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6 | Definit si le client a remboursé son crédit pour tel ou tel mois en tenant compte des mois de retard. |
| Montant dû| BILL_AMT1, BILL_AMT2, BILL_AMT3, BILL_AMT4, BILL_AMT5, BILL_AMT6 | Defini le montant total que le client devait payer pour un certain mois |
| Paiement précédent | PAY_AMT1, PAY_AMT2, PAY_AMT3, PAY_AMT4, PAY_AMT5, PAY_AMT6 | Donne le montant du precedent paiement du client |
| Paiement du mois prochain | default.payment.next.month | Dis si le client a payé ou pas son credit |

In [50]:
df["EDUCATION"].value_counts()
# Il faut separer les 468 dont on ne connait pas le niveau d'etudes du reste des donnees (51 'unknown', 280 'unknown', 123 'others' et 14 avec 0 dont on connait pas la signification.

,count
EDUCATION,
2,14030
1,10585
3,4917
5,280
4,123
6,51
0,14


In [51]:
df["MARRIAGE"].value_counts()
# Il faut separer les 377 dont on ne connait pas lea situation matrimoniale du reste des donnees (323 'others' et 54 avec 0 dont on connait pas la signification.

,count
MARRIAGE,
2,15964
1,13659
3,323
0,54


In [52]:
import sys
sys.path.append("..")   # Trouver le dossier src/

from src.data import nettoyer_donnees

df_propre = nettoyer_donnees(df)
df_propre["EDUCATION"].value_counts()   # le 0, le 5 et le 6 ont normalement disparu

,count
EDUCATION,
2,14030
1,10585
3,4917
4,468


In [53]:
df["EDUCATION"].value_counts()
# nottoyer_donnees a travaillé su une copie, donc le tableau d'origine est toujours present...

,count
EDUCATION,
2,14030
1,10585
3,4917
5,280
4,123
6,51
0,14


In [54]:
df_propre["MARRIAGE"].value_counts()

,count
MARRIAGE,
2,15964
1,13659
3,377


In [55]:
y = df_propre["default.payment.next.month"]
X = df_propre.drop(columns=["default.payment.next.month", "ID"])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # 20 % des clients gardés pour le test final
    random_state=11,   # fixe le hasard du découpage
    stratify=y,        # fixer le pourcentage de defauts a 22% dans chaque echantillon de donnees
)

In [56]:
X_train.shape

(24000, 23)

In [57]:
X_test.shape

(6000, 23)

In [58]:
y_train.value_counts(normalize=True)

,proportion
default.payment.next.month,
0,0.778792
1,0.221208


In [59]:
y_test.value_counts(normalize=True)

,proportion
default.payment.next.month,
0,0.778833
1,0.221167


In [60]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

paresseux = DummyClassifier(strategy="most_frequent")   # repondra toujours la valeur la plsu frequente
paresseux.fit(X_train, y_train)     # apprendre des donnees d'entrainement
predictions = paresseux.predict(X_test)     # predire

print(f"Accuracy : {accuracy_score(y_test, predictions):.2f}")
print(f"Recall : {recall_score(y_test, predictions):.2f}")

Accuracy : 0.78
Recall : 0.00


In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score

modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))  # max_iter : nombre d'essais pour trouver les bons points pour le bareme
modele.fit(X_train, y_train)    # apprendre des donnees d'entrainement
predictions = modele.predict(X_test)    # predire

print(f"Accuracy : {accuracy_score(y_test, predictions):.2f}")
print(f"Recall : {recall_score(y_test, predictions):.2f}")
print(f"Precision : {precision_score(y_test, predictions):.2f}")

Accuracy : 0.81
Recall : 0.23
Precision : 0.70


In [62]:
modele_equilibre = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced"),     # les mauvais payeurs comptent plus.
)
modele_equilibre.fit(X_train, y_train)
predictions = modele_equilibre.predict(X_test)

print(f"Accuracy : {accuracy_score(y_test, predictions):.2f}")
print(f"Recall : {recall_score(y_test, predictions):.2f}")
print(f"Precision : {precision_score(y_test, predictions):.2f}")

Accuracy : 0.69
Recall : 0.65
Precision : 0.38


| Modele | Accuracy | Recall | Precision |
| :--- | :--- | :--- | :--- |
| Regression logistique simple | 81% | 23% | 70% |
| Regression logistique avec poid de classe | 69% | 65% | 38% |


In [63]:
# Probabilité de défaut pour chaque client ([:, 1] = la colonne "défaut")
probas = modele.predict_proba(X_test)[:, 1]

# On applique notre propre seuil : True/False devient 1/0
seuils = [0.3, 0.4, 0.5]
mesures_reg_logistique = []
for seuil in seuils:
    predictions_seuil = (probas >= seuil).astype(int)
    mesures_reg_logistique.append({
      "Seuil": seuil,
      "Accuracy": accuracy_score(y_test, predictions_seuil),
      "Recall": recall_score(y_test, predictions_seuil),
      "Precision": precision_score(y_test, predictions_seuil),
  })
pd.DataFrame(mesures_reg_logistique)

,Seuil,Accuracy,Recall,Precision
0,0.3,0.802000,0.470234,0.562669
1,0.4,0.815833,0.372268,0.644909
2,0.5,0.808000,0.233610,0.696629


In [64]:
from sklearn.metrics import roc_auc_score

probas = modele.predict_proba(X_test)[:,1]
print(f"AUC Modele logistique : {roc_auc_score(y_test, probas):.2f}")
probas = modele_equilibre.predict_proba(X_test)[:,1]
print(f"AUC Modele Poids de classe : {roc_auc_score(y_test, probas):.2f}")

AUC Modele logistique : 0.72
AUC Modele Poids de classe : 0.72


> Note: `class_weight="balanced"` pousse toutes les probabilités vers le haut, donc le modèle accuse plus. Mais l’ordre des clients reste presque le même. Comme l’AUC ne juge que l’ordre, elle reste à 0,72. modele et modele_equilibre sont essentiellement les memes.

In [65]:
from sklearn.ensemble import RandomForestClassifier

foret = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
foret.fit(X_train, y_train)
predictions = foret.predict(X_test)
proba = foret.predict_proba(X_test)[:,1]
res = {
    "Modele": "Foret",
    "Accuracy": accuracy_score(y_test, predictions),
    "Recall": recall_score(y_test, predictions),
    "Precision": precision_score(y_test, predictions),
    "AUC": roc_auc_score(y_test, proba),
}
mesures_reg_logistique.append(res)
pd.DataFrame(mesures_reg_logistique)

,Seuil,Accuracy,Recall,Precision,Modele,AUC
0,0.3,0.802000,0.470234,0.562669,NaN,NaN
1,0.4,0.815833,0.372268,0.644909,NaN,NaN
2,0.5,0.808000,0.233610,0.696629,NaN,NaN
3,NaN,0.821500,0.384326,0.667539,Foret,0.775236


In [66]:
proba_train = foret.predict_proba(X_train)[:,1]
print(f"AUC entraînement : {roc_auc_score(y_train, proba_train):.2f}")

AUC entraînement : 1.00


In [67]:
res = []
valeurs = [1, 20, 50, 500, 3000]
for valeur in valeurs:
    foret = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, min_samples_leaf=valeur)
    foret.fit(X_train, y_train)
    proba = foret.predict_proba(X_test)[:,1]
    proba_train = foret.predict_proba(X_train)[:,1]
    res.append({
        "Valeur": valeur,
        "AUC de test": roc_auc_score(y_test, proba),
        "AUC d'entrainement": roc_auc_score(y_train, proba_train),
    })
    
pd.DataFrame(res)

,Valeur,AUC de test,AUC d'entrainement
0,1,0.775236,0.999992
1,20,0.787924,0.889704
2,50,0.789684,0.838846
3,500,0.783146,0.784078
4,3000,0.747794,0.740950
